# Physically Constraining SR-ALL

This notebook evaluates SR-ALL before and after enforcing physical constraints.

**Physical constraints (from `constraints.ipynb`):**
- PC3: $\partial P / \partial \widehat{\text{RH}} \geq 0$ (more moisture → more rain)
- PC4: $\partial P / \partial \widehat{\theta_e} \geq 0$ (more buoyancy → more rain)
- PC5: $\partial P / \partial \widehat{\theta_e^*} \leq 0$ (less stability → more rain)

**Key context:** These constraints are derivatives with respect to kernel-integrated
features, not the raw atmospheric profiles. That the constraints take such simple forms —
monotonic partial derivatives — is a direct consequence of the parametric (Gaussian) kernel
framework: the kernels compress vertical profiles into scalar features that precipitation
monotonically depends on. This is not a given; it is a notable property of the kernel
integration approach that makes physical constraints easy to state and enforce.

**Key finding from `constraints.ipynb`:** PC3 violations cluster in the low-moisture,
high-stability regime where precipitation is effectively zero. The negative $\partial P /
\partial \widehat{\text{RH}}$ slopes in these cubes are orders of magnitude weaker than the
positive slopes in the convectively active regime (mean |slope| = 0.002 violated vs. 1.48
satisfied). These violations are almost certainly noise artifacts in near-zero precipitation,
not a physically meaningful signal worth modeling.

**Approach:** Following Beucler et al., we enforce constraints as a post-hoc modification
of the top-performing SR equation, producing a "constrained" variant (SR-ALL-C). This
belongs in the evaluation/results section — since only one model is constrained, it does
not require a methods section.

In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import sympy as sp
import proplot as pplt
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

In [ ]:
regpath = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
with open(regpath,'rb') as f:
    REGISTRY = pickle.load(f)
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}
print(f'Loaded {len(ORDER)} optimized equations: {[LABELS[n] for n in ORDER]}')
for name in ORDER:
    entry = REGISTRY[name]
    cstr = ', '.join(f'{k}={v:.4f}' for k,v in entry['constants'].items())
    print(f'  {LABELS[name]}: {entry["form"]}  [{cstr}]')

## 1. Evaluate SR-ALL before constraints

In [ ]:
TARGETNAME = 'sr_all_eq'
assert TARGETNAME in REGISTRY, f'{TARGETNAME} not in registry — run optimizer first'

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
            'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

def predict_eq(name,columns):
    entry = REGISTRY[name]
    form,constants = entry['form'],entry['constants']
    raw = eval_form(form,columns,constants)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

cols = get_columns()
pred_unconstrained = predict_eq(TARGETNAME,cols)
raw_unconstrained  = eval_form(REGISTRY[TARGETNAME]['form'],cols,REGISTRY[TARGETNAME]['constants'])

r2_all   = 1 - np.mean((pred_unconstrained - obs)**2) / np.var(obs)
r2_land  = 1 - np.mean((pred_unconstrained[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2_ocean = 1 - np.mean((pred_unconstrained[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])
print(f'{LABELS[TARGETNAME]} (unconstrained):')
print(f'  R² all={r2_all:.4f}  land={r2_land:.4f}  ocean={r2_ocean:.4f}')

## 2. Constraint satisfaction before enforcement

In [ ]:
BASEVARS = ['rh','thetae','thetaestar','lf','shf','lhf','bl']
NCUBES = [3,4,5,6,7]

CONSTRAINTS = {
    'PC3':{
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1,
        'other_vars':['thetae','thetaestar']},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1,
        'other_vars':['rh','thetaestar']},
    'PC5':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1,
        'other_vars':['rh','thetae']}}

def cube_monotonicity_test(name,target_var,expected_sign,other_vars,ncubes,mask=None):
    cols = get_columns()
    targetvals = cols[target_var]
    othervalslist = [cols[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        nsel = sel.sum()
        x = targetvals[sel]
        overrides = {k:cols[k][sel] for k in BASEVARS}
        overrides[target_var] = x
        for v,binarr,edgearr in zip(other_vars,bins,edges):
            ci = cidx
            for j in range(nother-1,-1,-1):
                if other_vars[j] == v:
                    bi = ci % ncubes
                    break
                ci //= ncubes
            midpoint = 0.5 * (edgearr[bi] + edgearr[bi+1])
            overrides[v] = np.full(nsel,midpoint)
        cubecols = get_columns(**overrides)
        p = predict_eq(name,cubecols)
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,p) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

print(f'Constraint satisfaction for {LABELS[TARGETNAME]} (unconstrained):')
pre_results = {}
for pcname,pc in CONSTRAINTS.items():
    pre_results[pcname] = {}
    for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        pcts = []
        for n in NCUBES:
            sat,tot = cube_monotonicity_test(TARGETNAME,pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
            pcts.append(sat/max(tot,1)*100)
        pre_results[pcname][region] = np.mean(pcts)
        print(f'  {pcname} ({pc["label"]}) {region}: {pre_results[pcname][region]:.1f}%')

## 3. Analytical partial derivatives

Before enforcing constraints numerically, compute the symbolic partial derivatives of the
SR-ALL form. Because the equation is built from simple operations (cube, max, products,
sums), the partial derivatives have closed-form expressions that tell us *where* and *why*
violations occur.

This is a direct benefit of the kernel integration framework: because the kernels compress
vertical profiles into scalar features that precipitation monotonically depends on, the
physical constraints reduce to sign conditions on partial derivatives of a known algebraic
form — not numerical derivatives of a black-box function.

In [ ]:
entry = REGISTRY[TARGETNAME]
form = entry['form']
consts = entry['constants']
print(f'Form: P = zmin + max({form}, 0)')
print(f'Constants: {consts}')
print()
print('The prediction pipeline is: raw = f(x), then P = expm1((zmin + max(raw,0)) * std + mean)')
print('So dP/dx has the sign of d(raw)/dx wherever raw > 0 (precipitation regime).')
print('Where raw <= 0, P = expm1(zmin * std + mean) = 0, so the constraint is trivially satisfied.')
print()
print('Therefore, constraint violations can only occur where raw > 0 (active precipitation).')
print('We only need to check the sign of d(raw)/d(feature) in the raw > 0 regime.')

In [ ]:
# TODO: once the final sr_all_eq form is decided after rerunning PySR,
# compute symbolic partials here. The structure will depend on the
# discovered form, but for the current sr_atm_eq + correction structure:
#
# d(raw)/d(rh):
#   - In the RH-dominated regime (rh > d*thetae - e*thetaestar - f):
#     d(raw)/d(rh) = 3a * rh^2 + (correction terms from sr_all)
#     This is positive when rh > 0 and a > 0 — satisfied in the convective regime.
#   - In the buoyancy-dominated regime: d(raw)/d(rh) = (correction terms only)
#     Sign depends on the surface correction form.
#
# d(raw)/d(thetae):
#   - In the buoyancy-dominated regime:
#     d(raw)/d(thetae) = 3a*(d*thetae-e*thetaestar-f)^2 * d + (correction terms)
#     Positive when d > 0 — satisfied.
#   - In the RH-dominated regime: (correction terms only)
#
# d(raw)/d(thetaestar):
#   - In the buoyancy-dominated regime:
#     d(raw)/d(thetaestar) = 3a*(d*thetae-e*thetaestar-f)^2 * (-e) + (correction terms)
#     Negative when e > 0 — satisfied.
#   - In the RH-dominated regime: (correction terms only)
#
# UPDATE THIS CELL once the final SR-ALL form is known.
print('Symbolic partial derivatives — update after final SR-ALL form is decided.')

## 4. Enforce constraints: construct SR-ALL-C

Following the approach of Beucler et al.: make minimal modifications to the equation so
that all three partial derivative constraints are satisfied everywhere. The key insight is
that the max() in SR-ATM already creates two regimes, and the cube() ensures strong
nonlinearity. The constraints just need to hold within each regime.

The constrained equation gets a different name (SR-ALL-C) and can be evaluated alongside
the unconstrained version.

In [ ]:
# TODO: implement the constrained form once the final SR-ALL equation is known.
# The tweak will depend on the specific surface correction terms discovered by PySR.
#
# General strategy:
#   1. Identify which partial derivatives can go negative and under what conditions.
#   2. Apply minimal algebraic modifications (e.g., abs() on a coefficient, clamp on
#      a correction term) so the partial derivative is guaranteed non-negative.
#   3. Re-optimize constants under the constrained form.
#
# For example, if the surface correction is additive:
#   raw = sr_atm_eq + g(lf, shf, lhf, thetae)
# then d(raw)/d(rh) = d(sr_atm_eq)/d(rh), which depends only on the max() branch.
# In the RH-dominated branch: d(raw)/d(rh) = 3a*rh^2 >= 0 always (since a > 0).
# In the buoyancy branch: d(raw)/d(rh) = 0 + d(g)/d(rh) — if g doesn't depend on rh,
# this is zero and satisfied.
#
# The tricky constraint is typically d(raw)/d(thetae) in the RH-dominated regime,
# where the correction term might introduce a negative thetae dependence.
#
# FILL IN after rerunning PySR and choosing the final SR-ALL form.
print('Constrained form — implement after final SR-ALL equation.')

## 5. Compare unconstrained vs. constrained

In [ ]:
# TODO: after implementing the constrained form above, fill in this comparison.
#
# pred_constrained = predict_constrained(cols)
# r2c_all   = 1 - np.mean((pred_constrained - obs)**2) / np.var(obs)
# r2c_land  = 1 - np.mean((pred_constrained[landmask] - obs[landmask])**2) / np.var(obs[landmask])
# r2c_ocean = 1 - np.mean((pred_constrained[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])
#
# comparison = pd.DataFrame({
#     'Model':['SR-ALL','SR-ALL-C'],
#     'R² (all)':[r2_all,r2c_all],
#     'R² (land)':[r2_land,r2c_land],
#     'R² (ocean)':[r2_ocean,r2c_ocean],
# })
# comparison
print('Performance comparison — fill in after constrained form is implemented.')

In [ ]:
# TODO: constraint satisfaction for the constrained model.
# Should be 100% by construction — verify.
#
# post_results = {}
# for pcname,pc in CONSTRAINTS.items():
#     post_results[pcname] = {}
#     for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
#         pcts = []
#         for n in NCUBES:
#             sat,tot = cube_monotonicity_test_constrained(...)
#             pcts.append(sat/max(tot,1)*100)
#         post_results[pcname][region] = np.mean(pcts)
print('Constraint satisfaction (constrained) — verify 100% after implementation.')

## 6. Where do constraints change predictions?

Map the spatial distribution of prediction differences between SR-ALL and SR-ALL-C.
If the constraints only matter in the dry/stable regime (as `constraints.ipynb` suggests),
the differences should be small and concentrated over ocean regions with suppressed
convection.

In [ ]:
# TODO: spatial map of |P_constrained - P_unconstrained|, time-averaged.
# Also scatter of P_constrained vs P_unconstrained colored by regime (land/ocean).
#
# Expected result: differences are negligible (< 0.1 mm) and concentrated
# in the dry ocean regime where P ≈ 0 anyway.
print('Spatial difference map — fill in after constrained form.')

## 7. Discussion

### The kernel framework enables simple constraints

The physical constraints tested here (PC3–PC5) are partial derivatives of precipitation
with respect to kernel-integrated features $\widehat{\text{RH}}$, $\widehat{\theta_e}$,
and $\widehat{\theta_e^*}$. That these take the form of simple monotonicity conditions
is not automatic — it is a consequence of the parametric Gaussian kernel framework, which
compresses vertical profiles into scalar features that precipitation monotonically depends
on. With raw vertical profiles or free-form features, the constraints would be far more
complex and harder to enforce.

### Ocean dP/dRH violations are noise, not physics

PC3 ($\partial P / \partial \widehat{\text{RH}} \geq 0$) is satisfied 100% over land.
Over ocean, violations occur in the low-moisture, high-stability regime where:
- Mean precipitation is ~0.01 mm (effectively zero)
- Negative slopes are ~700× weaker than positive slopes (0.002 vs. 1.48)
- The thermodynamic regime suppresses deep convection entirely

These violations likely reflect tiny noise artifacts in near-zero precipitation fields,
not a physically meaningful non-monotonic relationship. Enforcing dP/dRH ≥ 0 prevents
the model from learning these artifacts — this is the "good" case where the constraint
improves interpretability without sacrificing skill on physically relevant samples.

### Constraining improves extrapolation without hurting interpolation

"Despite not being explicitly enforced during training, SR-ALL satisfies physical
constraints for X% of samples. Physically constraining the top-performing equation
(SR-ALL-C) ensures 100% constraint satisfaction with negligible change in R²
(Δ R² = ...), confirming that the violations lie outside the physically relevant regime."

The constrained model has better asymptotic behavior: in the limit of extreme dryness
or extreme stability, SR-ALL-C is guaranteed to produce physically sensible responses,
while SR-ALL could produce artifacts. This matters for climate projections under novel
thermodynamic conditions.